# 2 · Batch — run every image

The tutorial processed one image. Here we run the **same** `segment → measure`
pipeline over all 121 cross-sections and collect two tidy tables in `outputs/`:

- **`all_vessels.csv`** — one row per vessel (every detected vessel in every
  image), with the shape metrics plus `site` / `group` / `rainfall`.
- **`area_fraction.csv`** — one row per image: the fraction of the frame taken
  up by vessel lumens, again tagged with `site` / `group` / `rainfall`.

Notebook 3 reads both of these. Run this notebook once before opening it.

In [ ]:
import os
from pathlib import Path

if not Path("data/10x").exists() and Path("../data/10x").exists():
    os.chdir("..")
print("working directory:", Path.cwd())

In [ ]:
import cv2
import pandas as pd

import vessel_morphometry as vm

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

## Study parameters

This study keeps vessels with an **equivalent diameter of 11-120 µm**. The
`vessel_morphometry` package ships general-purpose defaults (`MIN_VESSEL_DIAMETER_UM = 0`
keeps every lumen), so we set the study's cutoffs here - explicit and up front,
not buried - and the measurement step uses them (diameters are converted to
pixels via each image's µm/px).

In [ ]:
from vessel_morphometry import config

config.MIN_VESSEL_DIAMETER_UM = 11.0    # small-vessel cutoff for this study
config.MAX_VESSEL_DIAMETER_UM = 120.0   # drop blobs above this (e.g. overexposed background)

### Calibration: pixels → microns

Real-world units come from the **scale bar** on each plate. Run the calibration
script once to measure every bar and write `outputs/calibration.csv`:

```
uv run python scripts/calibrate_dataset.py
```

We then look up each image's **µm/px**. The lookup prefers a hand-entered
`label_um_manual` (÷ `bar_px`), then the script's `um_per_px`, then the cohort
median. If `calibration.csv` is missing it falls back to the constant
`CALIBRATION_UM_PER_PX["10x"]` from `config.py` (which ships as `None`, leaving
`area_um2` as `NaN`) and warns.

Reading the printed labels uses OCR (the optional `[ocr]` extra,
`rapidocr-onnxruntime`); if it is not installed the bar lengths still calibrate,
because the calibration run found µm/px to be the **same across plates** (~1.167)
— so the constant applies to every image.

In [ ]:
from vessel_morphometry.calibrate import um_per_px_lookup

# Per-image µm/px from the calibration run; default is used for any image not in
# the table (and the config constant if calibration.csv is missing entirely).
um_per_px, default_um_per_px = um_per_px_lookup(
    "outputs/calibration.csv", fallback_constant=vm.CALIBRATION_UM_PER_PX["10x"])
print("calibrated images:", len(um_per_px), "| default µm/px:", default_um_per_px)

### Run the pipeline over every image

We loop over the images in sorted order so the output is reproducible, measure
each one, and stash both the per-vessel table and the per-image area fraction.

In [ ]:
paths = sorted(Path("data/10x").glob("*.png"))
print(f"{len(paths)} images to process\n")

per_vessel_frames = []
area_fraction_rows = []

for n, path in enumerate(paths, start=1):
    img = cv2.imread(str(path))
    channel = vm.choose_channel(img)   # cyan -> G-R, white -> min(G,B); recorded for QC
    s = um_per_px.get(path.stem, default_um_per_px)   # microns per pixel for this plate
    p = vm.Params(um_per_px=s)
    labels = vm.segment(img, p)
    df = vm.measure(labels, img.shape, p, source=path.stem)   # fills area_um2 + *_um
    per_vessel_frames.append(df)

    # one area-fraction row per image, tagged with the experiment metadata
    site, group, rainfall = vm.parse_name(path.stem)
    fraction = vm.vessel_area_fraction(labels)
    area_fraction_rows.append({
        "source_image": path.stem, "site": site, "group": group,
        "rainfall": rainfall, "vessel_fraction": fraction["vessel_fraction"],
        "vessel_px": fraction["vessel_px"], "image_px": fraction["image_px"],
        "channel": channel,
    })

    print(f"[{n:3d}/{len(paths)}] {path.stem:28s} -> {len(df):3d} vessels")

print("\ndone.")

### Save the two tables

In [ ]:
all_vessels = pd.concat(per_vessel_frames, ignore_index=True)
area_fraction = pd.DataFrame(area_fraction_rows)

all_vessels.to_csv(OUT / "all_vessels.csv", index=False)
area_fraction.to_csv(OUT / "area_fraction.csv", index=False)

print("all_vessels.csv :", all_vessels.shape, "->", OUT / "all_vessels.csv")
print("area_fraction.csv:", area_fraction.shape, "->", OUT / "area_fraction.csv")

### Quick sanity check

In [ ]:
n_images = len(paths)
n_with_vessels = all_vessels["source_image"].nunique()

print("images processed       :", n_images)
print("images with >= 1 vessel:", n_with_vessels,
      f"({n_images - n_with_vessels} detected none)")
print("total vessels          :", len(all_vessels))
print("vessels per image (median):",
      int(all_vessels.groupby("source_image").size().median()))
print("\nvessels by rainfall class:")
print(all_vessels["rainfall"].value_counts())

# area_fraction always has one row per image, even when a section has no vessels.
assert len(area_fraction) == n_images, "area_fraction should cover every image"
all_vessels.head()

### Two things to notice about the counts

- **Zero-vessel images.** An occasional section detects no vessels at all; it
  contributes no rows to `all_vessels.csv`, so that table may cover slightly
  fewer than 121 images, while `area_fraction.csv` always has one row per image.
- **How many vessels?** This run keeps **every** detected vessel (~12,000 total,
  a median of ~112 per image) using the package defaults `min_area_px=30`,
  `max_area_px=4000`. If you want a smaller, cleaner table, raise `min_area_px`
  via `vm.Params(...)` here — that is a tuning knob, not a change to the package
  formulas. Notebook 3 also trims to whole vessels in a sensible size window
  before any statistics.

In [ ]:
area_fraction.groupby("site")["vessel_fraction"].describe().round(4)

In [ ]:
import pandas as pd
counts = pd.read_csv("outputs/all_vessels.csv").groupby("source_image").size().sort_values()
print(counts.describe())
print("LOWEST:\n", counts.head(8))
print("HIGHEST:\n", counts.tail(8))